# Examine LLM chat-template rendering

For each candidate LLM, document how its chat template renders
a prompt — turn markers, the `\n\n` step separator, and BOS —
so the prompts the search pipeline builds are correct by
construction (M1, llm-prm-deep-dive).

> **Findings + decision:** see the project note
> `llm-prm-deep-dive/findings/llm-chat-templates.md`
> (vault) and `docs/decisions.md` 2026-06-13 (Prompting).
> Short version: the pipeline now uses each model's
> **native** template and preserves the `\n\n` step
> separator with strip-and-reappend; it no longer forces a
> single custom template on every model. This notebook is
> the evidence behind that decision — it renders both the
> native and the former custom path so the difference is
> visible, not silent.

> **Scope:** this notebook is about *template rendering*. The
> system-prompt content itself (and per-model system-prompt
> behavior) is the sibling notebook
> `examine_llm_system_prompt_v1.ipynb`.

**GPU:** not required — tokenizer-only, runs in seconds.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

from transformers import AutoTokenizer

from utils.configs import GenConfig, build_conv

In [ ]:
base_dir = '/groups/chichengz/tnn/datasets/'

model_names = [
    "Llama3.2-1B-Instruct",
    "Llama3.2-3B-Instruct",
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-7B-Instruct",
]

# GenConfig supplies the system prompt (render payload) and the
# former custom_chat_template, so we render the real prompt path.
config = GenConfig()
config.date_string = "Aug 1 2025"

# A short sample problem and a two-step assistant response that
# ends with the \n\n step separator (the case that can break).
sample_question = "What is 2 + 2?"
sample_response = (
    "## Step 1: Add the numbers\n"
    "2 + 2 = 4.\n\n"
    "## Step 2: State the answer\n"
    "Therefore, the final answer is: $\\boxed{4}$.\n\n"
)

## Native template fingerprints

Before rendering, hash each model's shipped `chat_template` to
see which models share one. Models with the same fingerprint
have byte-identical native templates, so their native-path
results below will match — and the table flags the day a new
model breaks the pattern (don't assume "same family = same
template" across generations/variants; verify here).

In [3]:
import hashlib
from collections import defaultdict

# Cache tokenizers (reused by the render loops below) and record
# each model's native template fingerprint + bos token.
tokenizers = {}
fingerprints = {}
bos_tokens = {}
groups = defaultdict(list)

for name in model_names:
    tok = AutoTokenizer.from_pretrained(
        base_dir + name, trust_remote_code=True,
    )
    tokenizers[name] = tok
    fp = hashlib.md5(
        (tok.chat_template or "").encode()
    ).hexdigest()[:10]
    fingerprints[name] = fp
    bos_tokens[name] = tok.bos_token
    groups[fp].append(name)

print(f"{'model':<24}{'native fp':>12}{'bos_token':>20}")
print('-' * 56)
for name in model_names:
    print(
        f"{name:<24}{fingerprints[name]:>12}"
        f"{str(bos_tokens[name]):>20}"
    )

print("\nShared native templates (same fingerprint):")
for fp, names in groups.items():
    if len(names) > 1:
        print(f"  {fp}: {', '.join(names)}")

model                      native fp           bos_token
--------------------------------------------------------
Llama3.2-1B-Instruct      0ce9ccbb6a   <|begin_of_text|>
Llama3.2-3B-Instruct      0ce9ccbb6a   <|begin_of_text|>
Qwen2.5-3B-Instruct       9a1b106583                None
Qwen2.5-7B-Instruct       9a1b106583                None

Shared native templates (same fingerprint):
  0ce9ccbb6a: Llama3.2-1B-Instruct, Llama3.2-3B-Instruct
  9a1b106583: Qwen2.5-3B-Instruct, Qwen2.5-7B-Instruct


## Helpers

`render` builds the conversation and applies a template along
the same call path as search: `continue_final_message=True`
when there is an assistant turn, so the prompt is left open for
the model to continue. `sep_survives` checks whether a trailing
`\n\n` in the assistant content is still present after
rendering.

In [4]:
def render(tokenizer, question, response, system_prompt,
           custom_template=None):
    """Render a built conversation, mirroring the search path.

    If custom_template is given it overrides the tokenizer's
    native chat_template (what the pipeline does when
    config.custom_chat_template is set). Returns the rendered
    string, or the exception repr if templating raises.
    """
    convs = [build_conv(question, response, system_prompt)]
    if custom_template is not None:
        tokenizer.chat_template = custom_template
    has_assistant = response != ""
    try:
        out = tokenizer.apply_chat_template(
            convs,
            add_generation_prompt=not has_assistant,
            continue_final_message=has_assistant,
            tokenize=False,
        )
        return out[0]
    except Exception as e:
        return f"<RAISED: {type(e).__name__}: {e}>"


def sep_survives(rendered, response):
    """True if a trailing \\n\\n in `response` is still present at
    the tail of the rendered prompt."""
    if not response.endswith("\n\n"):
        return None
    if rendered.startswith("<RAISED:"):
        return False
    return rendered.endswith("\n\n")

## Native templates

For each model, render with its own shipped chat template (no
override). Inspect the turn markers and whether the trailing
step separator survives.

In [6]:
native_sep = {}
native_bos = {}

for name in model_names:
    print(f"\n{'=' * 60}\n=== {name} (native template) ===\n{'=' * 60}")
    tok = tokenizers[name]
    rendered = render(
        tok, sample_question, sample_response,
        config.system_prompt, custom_template=None,
    )
    print(rendered)
    survived = sep_survives(rendered, sample_response)
    native_sep[name] = survived
    bos = bos_tokens[name]
    native_bos[name] = bool(bos) and rendered.startswith(bos)
    print(f"\n  trailing '\\n\\n' survives: {survived}")
    print(f"  starts with bos ({bos}): {native_bos[name]}")


=== Llama3.2-1B-Instruct (native template) ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 13 Jun 2026

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is 2 + 2?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

## Step 1: Add the numbers
2 + 2 = 4.

## Step 2: State the answer
Therefore, the final answer is: $\boxed{4}$.

  trailing '\n\

## Pipeline template (`GenConfig.custom_chat_template`)

The single hardcoded template applied to every model by the
search code. Watch the Qwen models in particular: their native
`<|im_start|>` format is replaced by Llama-style headers here.

In [7]:
custom_sep = {}
custom_bos = {}

for name in model_names:
    print(f"\n{'=' * 60}\n=== {name} (custom_chat_template) ===\n{'=' * 60}")
    tok = tokenizers[name]
    native_template = tok.chat_template      # save to restore after
    rendered = render(
        tok, sample_question, sample_response,
        config.system_prompt,
        custom_template=config.custom_chat_template,
    )
    tok.chat_template = native_template      # render() mutated it
    print(rendered)
    survived = sep_survives(rendered, sample_response)
    custom_sep[name] = survived
    bos = bos_tokens[name]
    custom_bos[name] = bool(bos) and rendered.startswith(bos)
    print(f"\n  trailing '\\n\\n' survives: {survived}")
    print(f"  starts with bos ({bos}): {custom_bos[name]}")


=== Llama3.2-1B-Instruct (custom_chat_template) ===
<|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 13 Jun 2026

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is 2 + 2?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

## Step 1: Add the numbers
2 + 2 = 4.

## Step 2: State the answer
Therefore, the final answer is: $\boxed{4}$.



  trailing '\n\n' survive

## Summary: separator survival and BOS

Columns (`native` = model's own template, `pipe` = the pipeline's
`custom_chat_template`):

- **`sep:`** — does the trailing step separator `\n\n` survive
  templating? `True` = preserved, `False` = trimmed.
- **`bos:`** — does the rendered prompt start with the model's
  BOS (beginning-of-sequence) token, e.g. `<|begin_of_text|>`?

`⚠ differs` marks models where native and pipeline disagree on
the separator — those rows are the finding, since the pipeline
path is what search actually sends.

In [8]:
hdr = (
    f"{'model':<24}{'sep:native':>12}{'sep:pipe':>10}"
    f"{'bos:native':>12}{'bos:pipe':>10}{'':>11}"
)
print(hdr)
print('-' * len(hdr))
for name in model_names:
    flag = "  ⚠ differs" if native_sep[name] != custom_sep[name] else ""
    print(
        f"{name:<24}{str(native_sep[name]):>12}{str(custom_sep[name]):>10}"
        f"{str(native_bos[name]):>12}{str(custom_bos[name]):>10}{flag:>11}"
    )

model                     sep:native  sep:pipe  bos:native  bos:pipe           
-------------------------------------------------------------------------------
Llama3.2-1B-Instruct           False      True        True     False  ⚠ differs
Llama3.2-3B-Instruct           False      True        True     False  ⚠ differs
Qwen2.5-3B-Instruct             True      True       False     False           
Qwen2.5-7B-Instruct             True      True       False     False           
